# nb03 — вход по истощению × порода

**Откуда постановка.** nb02: предсказание породы сортирует риск, но фейд в
точке +5% отрицателен даже у «безопасных» — потому что +5% это середина бега.
Три источника указывают на один механизм (старая линия 09_3, вход практиков
после остановки роста, nb02): **фейд должен входить после того, как рост
остановился.**

**Определения истощения (оба каузальные):**
- `stallN` — первые N минут без нового максимума после пересечения +5%
  (N = 10/15/30), вход следующим закрытием;
- `cend` — подтверждённый конец кластера (10 тихих минут), вход следующим закрытием.

**Протокол:** всё копание на DEV (walk-forward предсказания внутри
2024-07..2025-06), связка замораживается, VALID/TEST видят её один раз.
Фильтр породы — предсказания nb02, сделанные на пересечении +5%, которое
всегда РАНЬШЕ обоих входов (каузально чисто).

In [1]:
import sys; sys.path.insert(0, '.')
from _lab import *

E = pd.read_parquet('_out/pump_exhaust.parquet')
E['entry'] = pd.to_datetime(E['entry'], utc=True)
B = pd.read_parquet('_out/breed_preds.parquet')
B['entry'] = pd.to_datetime(B['entry'], utc=True)
L = pd.read_parquet('_out/pump_levels.parquet')[['sym','entry','fade05_30','fade05_60','fade05_240']]
L['entry'] = pd.to_datetime(L['entry'], utc=True)

X = E.merge(B[['sym','entry','pred','monster','win']], on=['sym','entry'], how='inner')
X = X.merge(L, on=['sym','entry'], how='left')
print('events with breed prediction:', len(X))
print(X.groupby('win').size().to_string())

events with breed prediction: 30064
win
TEST      9734
TRAIN     8830
VALID    11500


## 1. Три входа лоб-в-лоб (без фильтра породы)

Одни и те же пампы, один горизонт выхода. Если истощение — правильный механизм,
stall/cend должны быть систематически лучше пересечения +5%.

In [2]:
ENTRIES = [('crossing +5%','fade05'), ('stall10','stall10'), ('stall15','stall15'),
           ('stall30','stall30'), ('cluster end','cend')]
for hz in (60, 240):
    print(f'=== mean pnl %, exit +{hz}м ===')
    rows = []
    for name, pref in ENTRIES:
        r = {'вход': name}
        for w in ('TRAIN','VALID','TEST'):
            p = X[X.win == w][f'{pref}_{hz}'].dropna()
            r[w] = round(p.mean()*100, 2)
        rows.append(r)
    print(pd.DataFrame(rows).set_index('вход').to_string()); print()

=== mean pnl %, exit +60м ===
              TRAIN  VALID  TEST
вход                            
crossing +5%  -0.12  -0.89 -0.08
stall10       -0.25  -0.55 -0.04
stall15       -0.24  -0.66 -0.11
stall30       -0.17  -0.71 -0.01
cluster end   -1.16  -1.83 -1.25

=== mean pnl %, exit +240м ===
              TRAIN  VALID  TEST
вход                            
crossing +5%  -0.24  -1.31 -0.06
stall10       -0.35  -0.87 -0.07
stall15       -0.30  -0.86 -0.08
stall30       -0.35  -0.79  0.03
cluster end   -2.38  -3.29 -2.21



## 2. Истощение × порода: децили предсказания на DEV

Вопрос: даёт ли связка «вход по истощению + безопасная порода» положительный
фейд там, где nb02 в одиночку не смог? Смотрим только DEV.

In [3]:
DEV = X[X.entry < pd.Timestamp('2025-07-01', tz='UTC')].copy()
DEV['dec'] = pd.qcut(DEV.pred, 5, labels=False, duplicates='drop')
for pref in ('stall15','cend'):
    t = DEV.groupby('dec').agg(n=('pred','size'),
        monster=('monster', lambda s: round(s.mean()*100,0)),
        f60=(f'{pref}_60', lambda s: round(s.mean()*100,2)),
        f240=(f'{pref}_240', lambda s: round(s.mean()*100,2)),
        med240=(f'{pref}_240', lambda s: round(s.median()*100,2)))
    print(f'=== DEV, вход {pref}, квинтили P(monster) ===')
    print(t.to_string()); print()

=== DEV, вход stall15, квинтили P(monster) ===
        n  monster   f60  f240  med240
dec                                   
0    1766     12.0 -0.44 -0.51   -0.00
1    1766     17.0 -0.26 -0.33    0.10
2    1766     21.0 -0.13 -0.05    0.76
3    1766     26.0  0.06 -0.02    0.89
4    1766     33.0 -0.43 -0.60    0.76

=== DEV, вход cend, квинтили P(monster) ===
        n  monster   f60  f240  med240
dec                                   
0    1766     12.0 -0.39 -1.01   -0.51
1    1766     17.0 -1.08 -2.27   -1.91
2    1766     21.0 -1.48 -2.84   -2.22
3    1766     26.0 -1.19 -2.40   -1.60
4    1766     33.0 -1.67 -3.39   -2.15



## 3. Заморозка и единственный взгляд на VALID/TEST

Правило фиксируется по §1–2 на DEV (вход, горизонт, порог породы) и
применяется к VALID/TEST один раз, вместе со статистикой хвоста.

In [4]:
# --- параметры заморожены по DEV (см. вывод §1-2): заполняются после чтения DEV
PREF = 'stall15'; HZ = 60; THR = 0.20
def stats(p):
    p = p.dropna()
    if not len(p): return {}
    return dict(n=len(p), mean=round(p.mean()*100,2), med=round(p.median()*100,2),
                win=round((p>0).mean()*100,0), q10=round(p.quantile(0.1)*100,2))
print(f'правило: вход {PREF}, выход +{HZ}м, фейдим только pred < {THR}')
for w in ('TRAIN','VALID','TEST'):
    s = X[X.win == w]
    all_, flt = s[f'{PREF}_{HZ}'], s[s.pred < THR][f'{PREF}_{HZ}']
    print(f'{w}: все {stats(all_)}')
    print(f'{" "*len(w)}  фильтр {stats(flt)}')

правило: вход stall15, выход +60м, фейдим только pred < 0.2
TRAIN: все {'n': 8776, 'mean': np.float64(-0.24), 'med': np.float64(0.05), 'win': np.float64(51.0), 'q10': np.float64(-4.6)}
       фильтр {'n': 4958, 'mean': np.float64(-0.32), 'med': np.float64(-0.1), 'win': np.float64(49.0), 'q10': np.float64(-3.88)}
VALID: все {'n': 11475, 'mean': np.float64(-0.66), 'med': np.float64(-0.12), 'win': np.float64(49.0), 'q10': np.float64(-7.7)}
       фильтр {'n': 3288, 'mean': np.float64(-0.51), 'med': np.float64(-0.22), 'win': np.float64(46.0), 'q10': np.float64(-4.48)}
TEST: все {'n': 9713, 'mean': np.float64(-0.11), 'med': np.float64(0.25), 'win': np.float64(53.0), 'q10': np.float64(-6.92)}
      фильтр {'n': 2629, 'mean': np.float64(0.07), 'med': np.float64(0.25), 'win': np.float64(55.0), 'q10': np.float64(-3.77)}


## Выводы nb03

**1. 🟢 Вход по истощению НЕ спасает фейд.** stall10/15/30 ≈ то же, что вход на
пересечении +5% (−0.1…−0.9% по окнам), фильтр породы не переводит через ноль
(замороженное правило: TRAIN −0.32 / VALID −0.51 / TEST +0.07). Kill-критерий
семейства «одиночный шорт-фейд по кластерным событиям на минутках» выполнен:
закрыто тремя notebooks с трёх сторон (уровень nb02, истощение nb03, порода
nb02-03). Оставшаяся надежда фейда — микро-тайминг на секундах (коллектор),
это другой класс данных.

**2. 🟢 Сюрприз: у «конца кластера» ЗНАК ДРУГОЙ, большой и стабильный.**
Шорт на подтверждённом конце кластера теряет −1.2/−1.8/−1.3% (60м) и
−2.4/−3.3/−2.2% (240м) на TRAIN/VALID/TEST — то есть ЛОНГ там выигрывает
на ВСЕХ ТРЁХ окнах. Причём на DEV эффект монотонно растёт с P(monster):
лонг верхнего квинтиля +3.39%/240м против +1.01% нижнего. Это первый
кандидат v3, положительный на всех трёх окнах, и он совпадает с плейбуком
практиков (их G4 → LONG в журнале сделок).

**3. ⚠️ Что означает «−fade» для лонга — честно.** Колонки моделируют шорт
с 30% катастроф-стопом: «−fade» = лонг БЕЗ стопа, вход следующим закрытием,
без funding'а (а funding на разогнанных перпах обычно ПРОТИВ лонга и бывает
экстремальным). Хвост, стоп и funding лонга не смоделированы — п.2 это
УКАЗАТЕЛЬ, не результат.

**nb04: лонг-континуация после кластера, честно.** Свой стоп (по низу),
funding по факту, хвостовая статистика (q10, худшие дни, концентрация по
символам), перекрытия событий; порода как фильтр (верхние квинтили);
DEV → заморозка → VALID/TEST. Отдельно проверить: не является ли эффект
чистой бетой к альт-сезону (сравнить с лонгом случайной минуты той же монеты).

---

## ⛔ ПОПРАВКА (поймана при сборке nb04)

**§2 для входа cend был загрязнён lookahead'ом.** Предсказание породы живёт на
пересечении +5%, а оно случается ДО конца кластера лишь в ~17% событий (кластер
умирает за ~15м, до +5% цена в медиане идёт 35–112м). Для остальных 83% квинтильная
таблица cend×pred использовала информацию из будущего («памп ещё добежит до +5%»).

**Снимается:** монотонность лонга по P(monster) для входа cend (+1.0→+3.4).
**Остаётся в силе:** §1 полностью (безусловные входы, каузальны) — в т.ч. сам факт
«лонг на конце кластера положителен на всех трёх окнах»; §2 для stall-входов
(stall по построению после пересечения). nb04 обязан либо использовать породу
только на 17% подмножестве, либо строить признаки породы прямо на баре конца кластера.

## ⛔ ПОПРАВКА 2 (nb04, проверено вычислением)

Загрязнена была не только таблица §2, но и строка «cluster end» в §1: все таблицы
считались на событиях, слитых с breed_preds (inner join) = только пампы, ДОШЕДШИЕ
до +5%. Для cend-входа это условие из будущего (83% пересечений — после конца
кластера): строка означала «лонг при условии, что памп потом вырастет». nb04 на
ПОЛНОЙ книге: ride60 TEST −0.19% (а не +1.25); гипотеза же «дело в филе same-bar
vs next-bar» опровергнута прямым замером (+0.02pp). Строки stall/crossing из §1
каузальны (их входы — после пересечения). **Урок в копилку: lookahead прячется
и в ОТБОРЕ ВЫБОРКИ** — сэмпл, определённый будущим условием, портит даже
каузальные входы.